### Practica 1 - RA 1
**Alumno** : Gabriel Herrador
**Modulo** : M3 - Programació d'inteligencia artificial

### Paso 1: Carga y centralización de ficheros CSV y JSON en un diccionario
**Propósito:** Agrupar todos los conjuntos de datos en una estructura de diccionario centralizada. Esto nos permite mejorar la accesibilidad a los diferentes orígenes de datos, agilizar el análisis individual de cada estructura y preparar el terreno para aplicar una limpieza automatizada y conjunta de todos los ficheros.


In [6]:
import os
from pathlib import Path
import pandas as pd

# Directorio donde tengo los archivos de inventario es decir 
ruta_directorio = Path(".")

def cargar_inventario(nombre_archivo):
    archivo_path = Path(nombre_archivo)
    extension = archivo_path.suffix.lower()
    print(f"Procesando archivo: {nombre_archivo}")

    if extension == ".csv":

        df = pd.read_csv(archivo_path)
        print("--> Detectado como formato CSV (Tabular)")

    elif extension == ".json":
        df = pd.read_json(archivo_path)
        print("--> Detectado como formato JSON (Tabular)")
    else:
        raise ValueError(f"Formato de archivo no soportado: {extension}")

    return df

diccionario_inventario = {}

archivos = list(ruta_directorio.glob("*inventario_*.csv")) + list(ruta_directorio.glob("*inventario_*.json"))
for archivo in archivos:
    diccionario_inventario[archivo.name] = cargar_inventario(archivo)

print("\n Diccionario creado con exito")
print("Claves disponibles:" , list(diccionario_inventario.keys()))


Procesando archivo: inventario_BCN.csv
--> Detectado como formato CSV (Tabular)
Procesando archivo: inventario_MAD.csv
--> Detectado como formato CSV (Tabular)
Procesando archivo: inventario_SEV.json
--> Detectado como formato JSON (Tabular)
Procesando archivo: inventario_VAL.json
--> Detectado como formato JSON (Tabular)

 Diccionario creado con exito
Claves disponibles: ['inventario_BCN.csv', 'inventario_MAD.csv', 'inventario_SEV.json', 'inventario_VAL.json']


### Paso 2: Análisis detallado por columna (inventario_BCN.csv) y definición de errores
**Propósito:** Explorar en profundidad cada una de las columnas del conjunto de datos de Barcelona para detectar anomalías, valores nulos, inconsistencias o errores, con el fin de diseñar la lógica de limpieza posterior que se aplicará de forma automatizada al resto de ficheros.

### Paso 2 - Columna 1: Product_id
**Propósito:** Disponer de un identificador único que distinga de forma unívoca cada uno de los registros (filas) contenidos en los diferentes archivos de inventario.

**Problemas encontrados:** 
Aunque en el archivo de muestra (`inventario_BCN.csv`) la columna se encuentra completamente íntegra y sin duplicados, al auditar de forma global el resto de sedes se detectan anomalías puntuales (valores nulos y registros duplicados en algunos ficheros como Madrid o Sevilla).

**Solución:** 
Se establecerá una regla general en la función de limpieza automatizada para que, independientemente de que un dataset esté limpio o presente incidencias, se verifiquen y corrijan los valores nulos y duplicados del identificador en los cuatro ficheros del diccionario de manera simultánea.

In [96]:
# Seleccionamos el dataframe de BCN ya q es el q queremos trabajar e analizar 
df_bcn = diccionario_inventario["inventario_VAL.json"]

print("----Analisis de la Columna: Producto_id ----")


print("\n [1] Informacion basica y nulos")
print(df_bcn["producto_id"].info())
print(f"Cantidad de valores nulos: {df_bcn['producto_id'].isnull().sum()}")

duplicados = df_bcn["producto_id"].duplicated().sum()
print(f"Cantidad de Producto_id duplicados: {duplicados}")

print("\n Muestra de valores de Producto_id:")
print(df_bcn["producto_id"].head(10))

if df_bcn["producto_id"].dtype == "object":
    vacios_str = (df_bcn["producto_id"].str.strip() == "").sum()
    print(f"Cantidad de Producto_id vacios (string vacio): {vacios_str}")

----Analisis de la Columna: Producto_id ----

 [1] Informacion basica y nulos
<class 'pandas.Series'>
RangeIndex: 750 entries, 0 to 749
Series name: producto_id
Non-Null Count  Dtype
--------------  -----
750 non-null    str  
dtypes: str(1)
memory usage: 12.6 KB
None
Cantidad de valores nulos: 0
Cantidad de Producto_id duplicados: 0

 Muestra de valores de Producto_id:
0    VAL-00001
1    VAL-00002
2    VAL-00003
3    VAL-00004
4    VAL-00005
5    VAL-00006
6    VAL-00007
7    VAL-00008
8    VAL-00009
9    VAL-00010
Name: producto_id, dtype: str


### Paso 2 - Columna 2: nombre
**Propósito:** Almacenar la denominación o descripción comercial del producto del inventario para su correcta identificación en el catálogo.

**Problemas encontrados:** 
En el archivo de muestra (`inventario_BCN.csv`), la columna se presenta completa (750 registros sin valores `NaN`). No obstante, en un entorno multi-sede es común encontrar inconsistencias como espacios en blanco al inicio/final, caracteres especiales descontrolados o diferencias de capitalización (mayúsculas/minúsculas).

**Solución:** 
Se incorporará dentro de la función de limpieza masiva una regla estandarizada para la columna `nombre` que aplique eliminación de espacios innecesarios (`strip()`) y homogenización del formato de texto para todos los datasets del diccionario, garantizando la calidad del dato sin importar la sede de origen.

In [97]:

columna_2 = df_bcn.columns[1]

print(f"--- Analisis de la Columna 2: {columna_2} ---")


# 1. Información básica, tipo de datos y nulos
print("\n[1] Información básica y nulos:")
print(df_bcn[columna_2].info())
print(f"Cantidad de valores nulos (NaN): {df_bcn[columna_2].isnull().sum()}")

# 2. Comprobar valores vacíos o espacios en blanco si es de tipo texto (object)
if df_bcn[columna_2].dtype == 'object':
    vacios_str = (df_bcn[columna_2].astype(str).str.strip() == "").sum()
    print(f"Cantidad de valores vacíos/espacios en blanco: {vacios_str}")

# 3. Mostrar una muestra de los datos para ver qué tipo de información contiene
print(f"\nMuestra de valores de {columna_2}:")
print(df_bcn[columna_2].head(10))

# 4. Estadísticas básicas (si es numérico) o conteo de valores frecuentes (si es texto)
print(f"\nResumen de contenido:")
if pd.api.types.is_numeric_dtype(df_bcn[columna_2]):
    print(df_bcn[columna_2].describe())
else:
    print(df_bcn[columna_2].value_counts().head(5))

--- Analisis de la Columna 2: nombre ---

[1] Información básica y nulos:
<class 'pandas.Series'>
RangeIndex: 750 entries, 0 to 749
Series name: nombre
Non-Null Count  Dtype
--------------  -----
750 non-null    str  
dtypes: str(1)
memory usage: 19.3 KB
None
Cantidad de valores nulos (NaN): 0

Muestra de valores de nombre:
0              Cámara IP NX-115
1          Hub domótico Max-780
2               DAC USB Neo-500
3               Volante Neo-471
4               Hub USB Air-358
5                Sensor Pro-236
6         Enchufe smart Pro-371
7              Cargador Neo-182
8         Enchufe smart Pro-898
9    Protector pantalla Pro-912
Name: nombre, dtype: str

Resumen de contenido:
nombre
Interfaz audio Pro-737    2
Powerbank Pro-234         2
Barra de sonido Go-977    2
Cámara IP NX-115          1
Hub domótico Max-780      1
Name: count, dtype: int64


### Paso 2 - Columna 3: categoria
**Propósito:** Clasificar los productos del inventario dentro de agrupaciones temáticas o sectores comerciales predefinidos (ej. *Gaming*, *Telefonía*, *Informática*).

**Problemas encontrados:** 
Aunque en el archivo de muestra (`inventario_BCN.csv`) la columna se encuentra completa y sin valores nulos, en el conjunto global de los ficheros de inventario este tipo de campos de texto suele presentar inconsistencias menores, como variaciones en acentos, espacios sobrantes al inicio o final, o ligeras disparidades tipográficas entre sedes.

**Solución:** 
Se incluirá en la función de limpieza masiva una normalización estandarizada para la columna `categoria` (aplicando limpieza de espacios y estandarización de formato de texto) de forma simultánea en los cuatro datasets del diccionario, asegurando la consistencia de las agrupaciones para los informes y análisis posteriores.

In [98]:
columna_3 = df_bcn.columns[2]

print(f"--- Analisis de la Columna 3: {columna_3} ---")
print("\n[1] Información básica y nulos:")
print(df_bcn[columna_3].info())
print(f"Cantidad de valores nulos (NaN): {df_bcn[columna_3].isnull().sum()}")

# 2. Comprobación de espacios vacíos si es de tipo texto
if df_bcn[columna_3].dtype == 'object':
    vacios_str = (df_bcn[columna_3].astype(str).str.strip() == "").sum()
    print(f"Cantidad de valores vacíos/espacios en blanco: {vacios_str}")

# 3. Muestra de los datos
print(f"\nMuestra de valores de {columna_3}:")
print(df_bcn[columna_3].head(10))

# 4. Resumen estadístico (si es numérico) o frecuencias (si es texto)
print(f"\nResumen de contenido:")
if pd.api.types.is_numeric_dtype(df_bcn[columna_3]):
    print(df_bcn[columna_3].describe())
else:
    print(df_bcn[columna_3].value_counts().head(5))

--- Analisis de la Columna 3: categoria ---

[1] Información básica y nulos:
<class 'pandas.Series'>
RangeIndex: 750 entries, 0 to 749
Series name: categoria
Non-Null Count  Dtype
--------------  -----
750 non-null    str  
dtypes: str(1)
memory usage: 13.3 KB
None
Cantidad de valores nulos (NaN): 0

Muestra de valores de categoria:
0    Hogar inteligente
1    Hogar inteligente
2                Audio
3               Gaming
4          Informática
5    Hogar inteligente
6    Hogar inteligente
7            Telefonía
8    Hogar inteligente
9           Accesorios
Name: categoria, dtype: str

Resumen de contenido:
categoria
Gaming               140
Informática          128
Hogar inteligente    124
Audio                120
Telefonía            116
Name: count, dtype: int64


### Paso 2 - Columna 4: proveedor
**Propósito:** Identificar la entidad, distribuidor o fabricante responsable del suministro de cada producto para la gestión de compras y garantías.

**Problemas encontrados:** 
En el archivo de muestra (`inventario_BCN.csv`), la columna no presenta valores nulos (`NaN`). Sin embargo, al tratarse de nombres comerciales de proveedores ingresados en distintas sedes, existe el riesgo de incoherencias por espacios sobrantes, diferencias en mayúsculas/minúsculas o inconsistencias tipográficas entre ficheros.

**Solución:** 
Se incluirá en la función de limpieza masiva el tratamiento de la columna `proveedor` mediante la eliminación de espacios en blanco al inicio y al final (`strip()`) y la estandarización de formato de cadena, garantizando que el nombre del proveedor sea idéntico y homogéneo en los cuatro conjuntos de datos.

In [99]:
columna_4 = df_bcn.columns[3]

print(f"--- ANÁLISIS DE LA COLUMNA 4: {columna_4} ---")

# 1. Información básica, tipo de datos y nulos
print("\n[1] Información básica y nulos:")
print(df_bcn[columna_4].info())
print(f"Cantidad de valores nulos (NaN): {df_bcn[columna_4].isnull().sum()}")

# 2. Comprobación de espacios vacíos si es de tipo texto
if df_bcn[columna_4].dtype == 'object':
    vacios_str = (df_bcn[columna_4].astype(str).str.strip() == "").sum()
    print(f"Cantidad de valores vacíos/espacios en blanco: {vacios_str}")

# 3. Muestra de los datos
print(f"\nMuestra de valores de {columna_4}:")
print(df_bcn[columna_4].head(10))

# 4. Resumen estadístico (si es numérico) o frecuencias (si es texto)
print(f"\nResumen de contenido:")
if pd.api.types.is_numeric_dtype(df_bcn[columna_4]):
    print(df_bcn[columna_4].describe())
else:
    print(df_bcn[columna_4].value_counts().head(5))

--- ANÁLISIS DE LA COLUMNA 4: proveedor ---

[1] Información básica y nulos:
<class 'pandas.Series'>
RangeIndex: 750 entries, 0 to 749
Series name: proveedor
Non-Null Count  Dtype
--------------  -----
750 non-null    str  
dtypes: str(1)
memory usage: 15.2 KB
None
Cantidad de valores nulos (NaN): 0

Muestra de valores de proveedor:
0           SmartHub Europe
1                  OmniTech
2    ByteWorks Distribution
3                  OmniTech
4                  OmniTech
5               ElectroPlus
6                GigaSupply
7               ElectroPlus
8                  NextWare
9    ByteWorks Distribution
Name: proveedor, dtype: str

Resumen de contenido:
proveedor
ByteWorks Distribution    100
ElectroPlus                94
GigaSupply                 93
DigitalCore                84
NextWare                   80
Name: count, dtype: int64


### Paso 2 - Columna 5: precio_unitario
**Propósito:** Almacenar el valor económico unitario de cada producto en formato numérico decimal (`float`) para permitir cálculos de valoración de stock y análisis financieros del inventario.

**Problemas encontrados:** 
En el archivo de muestra (`inventario_BCN.csv`), la columna se encuentra completa, sin valores nulos (`NaN`) y con el tipo de dato correcto. No obstante, en cargas procedentes de múltiples sedes, este tipo de campos numéricos son propensos a contener anomalías como precios negativos, ceros ilógicos o posibles valores nulos en otros ficheros del diccionario.

**Solución:** 
Se incorporará en la función de limpieza masiva una regla de validación para la columna `precio_unitario` que controle y depure valores atípicos, negativos o nulos, aplicando un criterio estandarizado de corrección o imputación de forma simultánea en los cuatro datasets.

In [100]:
columna_5 = df_bcn.columns[4]


print(f"--- ANÁLISIS DE LA COLUMNA 5: {columna_5} ---")

# 1. Información básica, tipo de datos y nulos
print("\n[1] Información básica y nulos:")
print(df_bcn[columna_5].info())
print(f"Cantidad de valores nulos (NaN): {df_bcn[columna_5].isnull().sum()}")

# 2. Comprobación de espacios vacíos si es de tipo texto
if df_bcn[columna_5].dtype == 'object':
    vacios_str = (df_bcn[columna_5].astype(str).str.strip() == "").sum()
    print(f"Cantidad de valores vacíos/espacios en blanco: {vacios_str}")

# 3. Muestra de los datos
print(f"\nMuestra de valores de {columna_5}:")
print(df_bcn[columna_5].head(10))

# 4. Resumen estadístico (si es numérico) o frecuencias (si es texto)
print(f"\nResumen de contenido:")
if pd.api.types.is_numeric_dtype(df_bcn[columna_5]):
    print(df_bcn[columna_5].describe())
else:
    print(df_bcn[columna_5].value_counts().head(5))

--- ANÁLISIS DE LA COLUMNA 5: precio_unitario ---

[1] Información básica y nulos:
<class 'pandas.Series'>
RangeIndex: 750 entries, 0 to 749
Series name: precio_unitario
Non-Null Count  Dtype  
--------------  -----  
750 non-null    float64
dtypes: float64(1)
memory usage: 6.0 KB
None
Cantidad de valores nulos (NaN): 0

Muestra de valores de precio_unitario:
0    1196.17
1     579.61
2     469.73
3    1265.17
4     595.91
5    1381.71
6     553.22
7    1402.04
8    1181.19
9      17.05
Name: precio_unitario, dtype: float64

Resumen de contenido:
count     750.000000
mean      762.412813
std       439.514619
min         9.720000
25%       372.337500
50%       744.205000
75%      1151.020000
max      1497.700000
Name: precio_unitario, dtype: float64


### Paso 2 - Columna 6: stock_actual
**Propósito:** Registrar el volumen de unidades disponibles actualmente en el almacén en formato numérico entero para el control de stock y la planificación logística.

**Problemas encontrados:** 
En el archivo de muestra (`inventario_BCN.csv`), la columna se encuentra íntegra, con 750 registros completos, sin valores nulos (`NaN`) y con el tipo de dato adecuado[cite: 9]. No obstante, en un análisis global multisede, este tipo de campos son propensos a registrar anomalías como stocks negativos, nulos o valores atípicos.

**Solución:** 
Se incorporará en la función de limpieza masiva una regla de validación para la columna `stock_actual` que detecte y corrija cualquier valor negativo o nulo, aplicando el filtro de manera simultánea en los cuatro datasets del diccionario.

In [101]:
# Identificamos el nombre de la sexta columna automáticamente
columna_6 = df_bcn.columns[5]

print(f"--- ANÁLISIS DE LA COLUMNA 6: {columna_6} ---")

# 1. Información básica, tipo de datos y nulos
print("\n[1] Información básica y nulos:")
print(df_bcn[columna_6].info())
print(f"Cantidad de valores nulos (NaN): {df_bcn[columna_6].isnull().sum()}")

# 2. Comprobación de espacios vacíos si es de tipo texto
if df_bcn[columna_6].dtype == 'object':
    vacios_str = (df_bcn[columna_6].astype(str).str.strip() == "").sum()
    print(f"Cantidad de valores vacíos/espacios en blanco: {vacios_str}")

# 3. Muestra de los datos
print(f"\nMuestra de valores de {columna_6}:")
print(df_bcn[columna_6].head(10))

# 4. Resumen estadístico (si es numérico) o frecuencias (si es texto)
print(f"\nResumen de contenido:")
if pd.api.types.is_numeric_dtype(df_bcn[columna_6]):
    print(df_bcn[columna_6].describe())
else:
    print(df_bcn[columna_6].value_counts().head(5))

--- ANÁLISIS DE LA COLUMNA 6: stock_actual ---

[1] Información básica y nulos:
<class 'pandas.Series'>
RangeIndex: 750 entries, 0 to 749
Series name: stock_actual
Non-Null Count  Dtype
--------------  -----
750 non-null    int64
dtypes: int64(1)
memory usage: 6.0 KB
None
Cantidad de valores nulos (NaN): 0

Muestra de valores de stock_actual:
0    100
1     25
2      7
3     22
4     24
5     28
6     18
7     54
8     87
9     46
Name: stock_actual, dtype: int64

Resumen de contenido:
count    750.000000
mean      42.872000
std       31.270466
min        0.000000
25%       17.000000
50%       35.000000
75%       62.000000
max      156.000000
Name: stock_actual, dtype: float64


### Paso 2 - Columna 7: stock_minimo
**Propósito:** Definir el umbral mínimo de unidades que debe haber en el almacén para un producto determinado, permitiendo activar alertas tempranas de reabastecimiento y evitar roturas de stock.

**Problemas encontrados:** 
En el archivo de muestra (`inventario_BCN.csv`), la columna se encuentra íntegra, con 750 registros completos, sin valores nulos (`NaN`) y con un tipo de dato numérico adecuado[cite: 10]. Sin embargo, al extender la validación al conjunto global de ficheros, existe la posibilidad de encontrar valores negativos o nulos que invaliden las reglas de reposición.

**Solución:** 
Se incorporará en la función de limpieza masiva una validación específica para la columna `stock_minimo` que compruebe y corrija cualquier anomalía numérica o valor nulo de manera automatizada y simultánea en los cuatro datasets.

In [102]:
# Identificamos el nombre de la séptima columna automáticamente
columna_7 = df_bcn.columns[6]

print(f"--- ANÁLISIS DE LA COLUMNA 7: {columna_7} ---")

# 1. Información básica, tipo de datos y nulos
print("\n[1] Información básica y nulos:")
print(df_bcn[columna_7].info())
print(f"Cantidad de valores nulos (NaN): {df_bcn[columna_7].isnull().sum()}")

# 2. Comprobación de espacios vacíos si es de tipo texto
if df_bcn[columna_7].dtype == 'object':
    vacios_str = (df_bcn[columna_7].astype(str).str.strip() == "").sum()
    print(f"Cantidad de valores vacíos/espacios en blanco: {vacios_str}")

# 3. Muestra de los datos
print(f"\nMuestra de valores de {columna_7}:")
print(df_bcn[columna_7].head(10))

# 4. Resumen estadístico (si es numérico) o frecuencias (si es texto)
print(f"\nResumen de contenido:")
if pd.api.types.is_numeric_dtype(df_bcn[columna_7]):
    print(df_bcn[columna_7].describe())
else:
    print(df_bcn[columna_7].value_counts().head(5))

--- ANÁLISIS DE LA COLUMNA 7: stock_minimo ---

[1] Información básica y nulos:
<class 'pandas.Series'>
RangeIndex: 750 entries, 0 to 749
Series name: stock_minimo
Non-Null Count  Dtype
--------------  -----
750 non-null    int64
dtypes: int64(1)
memory usage: 6.0 KB
None
Cantidad de valores nulos (NaN): 0

Muestra de valores de stock_minimo:
0    34
1    19
2     4
3    11
4     9
5     9
6    15
7    16
8    39
9    38
Name: stock_minimo, dtype: int64

Resumen de contenido:
count    750.000000
mean      21.361333
std       10.976433
min        3.000000
25%       12.000000
50%       21.000000
75%       31.000000
max       40.000000
Name: stock_minimo, dtype: float64


### Paso 2 - Columna 8: almacen
**Propósito:** Indicar la ubicación física o centro logístico de procedencia de los productos (ej. Barcelona, Madrid, Sevilla, Valencia) para la trazabilidad multisede del inventario.

**Problemas encontrados:** 
En el archivo de muestra (`inventario_BCN.csv`), la columna está completa (750 registros con el valor "Barcelona") y sin nulos[cite: 11]. No obstante, al fusionar los datos de las distintas sedes, este campo puede presentar inconsistencias de formato, espacios en blanco o discrepancias tipográficas en la denominación de los almacenes.

**Solución:** 
Se incluirá en la función de limpieza masiva la validación y estandarización del campo `almacen` para garantizar una nomenclatura homogénea y limpia en todos los datasets del diccionario centralizado.

In [103]:
# Identificamos el nombre de la octava columna automáticamente
columna_8 = df_bcn.columns[7]

print(f"--- ANÁLISIS DE LA COLUMNA 8: {columna_8} ---")

# 1. Información básica, tipo de datos y nulos
print("\n[1] Información básica y nulos:")
print(df_bcn[columna_8].info())
print(f"Cantidad de valores nulos (NaN): {df_bcn[columna_8].isnull().sum()}")

# 2. Comprobación de espacios vacíos si es de tipo texto
if df_bcn[columna_8].dtype == 'object':
    vacios_str = (df_bcn[columna_8].astype(str).str.strip() == "").sum()
    print(f"Cantidad de valores vacíos/espacios en blanco: {vacios_str}")

# 3. Muestra de los datos
print(f"\nMuestra de valores de {columna_8}:")
print(df_bcn[columna_8].head(10))

# 4. Resumen estadístico (si es numérico) o frecuencias (si es texto)
print(f"\nResumen de contenido:")
if pd.api.types.is_numeric_dtype(df_bcn[columna_8]):
    print(df_bcn[columna_8].describe())
else:
    print(df_bcn[columna_8].value_counts().head(5))

--- ANÁLISIS DE LA COLUMNA 8: almacen ---

[1] Información básica y nulos:
<class 'pandas.Series'>
RangeIndex: 750 entries, 0 to 749
Series name: almacen
Non-Null Count  Dtype
--------------  -----
750 non-null    str  
dtypes: str(1)
memory usage: 11.8 KB
None
Cantidad de valores nulos (NaN): 0

Muestra de valores de almacen:
0    Valencia
1    Valencia
2    Valencia
3    Valencia
4    Valencia
5    Valencia
6    Valencia
7    Valencia
8    Valencia
9    Valencia
Name: almacen, dtype: str

Resumen de contenido:
almacen
Valencia    750
Name: count, dtype: int64


### Paso 2 - Columna 9: fecha_ultima_entrada
**Propósito:** Registrar la fecha de la última recepción o entrada de stock de cada producto en formato temporal para permitir análisis de antigüedad, rotación y caducidad de inventario.

**Problemas encontrados:** 
En el archivo de muestra (`inventario_BCN.csv`), la columna está completa (750 registros) y sin valores nulos (`NaN`)[cite: 12]. No obstante, al encontrarse como tipo texto (`str`), puede presentar inconsistencias de formato (ej. barras inclinadas en lugar de guiones), fechas inválidas o valores nulos en otras sedes del conjunto multisede.

**Solución:** 
Se incorporará en la función de limpieza masiva la conversión estandarizada de este campo a formato datetime de Pandas (`pd.to_datetime`), gestionando de forma automatizada los errores de formato y valores nulos de manera idéntica en los cuatro datasets del diccionario.

In [104]:
# Identificamos el nombre de la novena columna automáticamente
columna_9 = df_bcn.columns[8]

print(f"--- ANÁLISIS DE LA COLUMNA 9: {columna_9} ---")

# 1. Información básica, tipo de datos y nulos
print("\n[1] Información básica y nulos:")
print(df_bcn[columna_9].info())
print(f"Cantidad de valores nulos (NaN): {df_bcn[columna_9].isnull().sum()}")

# 2. Comprobación de espacios vacíos si es de tipo texto
if df_bcn[columna_9].dtype == 'object':
    vacios_str = (df_bcn[columna_9].astype(str).str.strip() == "").sum()
    print(f"Cantidad de valores vacíos/espacios en blanco: {vacios_str}")

# 3. Muestra de los datos
print(f"\nMuestra de valores de {columna_9}:")
print(df_bcn[columna_9].head(10))

# 4. Resumen estadístico (si es numérico) o frecuencias (si es texto)
print(f"\nResumen de contenido:")
if pd.api.types.is_numeric_dtype(df_bcn[columna_9]):
    print(df_bcn[columna_9].describe())
else:
    print(df_bcn[columna_9].value_counts().head(5))

--- ANÁLISIS DE LA COLUMNA 9: fecha_ultima_entrada ---

[1] Información básica y nulos:
<class 'pandas.Series'>
RangeIndex: 750 entries, 0 to 749
Series name: fecha_ultima_entrada
Non-Null Count  Dtype
--------------  -----
750 non-null    str  
dtypes: str(1)
memory usage: 13.3 KB
None
Cantidad de valores nulos (NaN): 0

Muestra de valores de fecha_ultima_entrada:
0    2026-09-06
1    2026-02-23
2    2026-01-11
3    2025-07-28
4    2025-08-04
5    2026-08-02
6    2026-08-04
7    2026-05-20
8    2026-07-12
9    2025-12-15
Name: fecha_ultima_entrada, dtype: str

Resumen de contenido:
fecha_ultima_entrada
2026-01-07    7
2026-05-04    6
2026-09-26    6
2026-04-29    6
2026-08-23    6
Name: count, dtype: int64


### Paso 2 - Columna 10: estado
**Propósito:** Especificar la situación operativa actual de cada producto dentro del catálogo (ej. *activo*, *descatalogado*) para filtrar referencias disponibles en la gestión de stock.

**Problemas encontrados:** 
En el archivo de muestra (`inventario_BCN.csv`), la columna no presenta valores nulos (`NaN`). Sin embargo, en el análisis de frecuencias se detectan inconsistencias críticas en el formato de texto debidas a la falta de estandarización en la capitalización (mezcla de minúsculas y mayúsculas como *activo*, *ACTIVO*, *descatalogado*, *DESCATALOGADO*).

**Solución:** 
Se incorporará en la función de limpieza masiva una regla de normalización estandarizada para el campo `estado` (aplicando conversión a minúsculas mediante `.str.lower()` y limpieza de espacios), asegurando la unificación de los valores categóricos de forma idéntica en los cuatro datasets.

In [105]:
# Identificamos el nombre de la décima columna automáticamente
columna_10 = df_bcn.columns[9]

print(f"--- ANÁLISIS DE LA COLUMNA 10: {columna_10} ---")

# 1. Información básica, tipo de datos y nulos
print("\n[1] Información básica y nulos:")
print(df_bcn[columna_10].info())
print(f"Cantidad de valores nulos (NaN): {df_bcn[columna_10].isnull().sum()}")

# 2. Comprobación de espacios vacíos si es de tipo texto
if df_bcn[columna_10].dtype == 'object':
    vacios_str = (df_bcn[columna_10].astype(str).str.strip() == "").sum()
    print(f"Cantidad de valores vacíos/espacios en blanco: {vacios_str}")

# 3. Muestra de los datos
print(f"\nMuestra de valores de {columna_10}:")
print(df_bcn[columna_10].head(10))

# 4. Resumen estadístico (si es numérico) o frecuencias (si es texto)
print(f"\nResumen de contenido:")
if pd.api.types.is_numeric_dtype(df_bcn[columna_10]):
    print(df_bcn[columna_10].describe())
else:
    print(df_bcn[columna_10].value_counts().head(5))

--- ANÁLISIS DE LA COLUMNA 10: estado ---

[1] Información básica y nulos:
<class 'pandas.Series'>
RangeIndex: 750 entries, 0 to 749
Series name: estado
Non-Null Count  Dtype
--------------  -----
750 non-null    str  
dtypes: str(1)
memory usage: 10.6 KB
None
Cantidad de valores nulos (NaN): 0

Muestra de valores de estado:
0    activo
1    activo
2    activo
3    activo
4    activo
5    activo
6    activo
7    activo
8    activo
9    activo
Name: estado, dtype: str

Resumen de contenido:
estado
activo           708
descatalogado     36
ACTIVO             6
Name: count, dtype: int64


###  Paso 3: Pipeline de Limpieza Automatizada Multisede

Tras completar la auditoría individual de los datasets (**BCN, MAD, SEV y VAL**), implementamos la función `limpiar_inventario()`. 

Esta función recorre el diccionario de sedes y aplica una serie de reglas de negocio automatizadas para corregir las inconsistencias detectadas:

* **`product_id`**: Eliminación de filas sin identificador (`NaN`) y depuración de registros duplicados (`keep='first'`).
* **`nombre` / `proveedor`**: Imputación de nulos con valores descriptivos por defecto y eliminación de espacios en blanco sobrantes (`.str.strip()`).
* **`categoria`**: Homogeneización del texto en formato título (`.str.capitalize()`) e imputación de nulos.
* **`precio_unitario`**: Conversión forzada a numérico (`pd.to_numeric`), corrección de valores negativos mediante su valor absoluto (`.abs()`), rellenado de nulos con la mediana e imputación a 2 decimales.
* **`stock_actual` / `stock_minimo`**: Estandarización a tipo entero (`int64`), corrección de valores negativos y rellenado de faltantes con la mediana.
* **`fecha_ultima_entrada`**: Conversión a tipo temporal nativo (`pd.to_datetime`).
* **`estado`**: Mapeo y normalización categórica en minúsculas (`activo` / `descatalogado`), corrigiendo ruido de datos como `'X'`, `'retirado?'` o `'pendiente'`.

In [108]:
import pandas as pd
import numpy as np

def limpiar_inventario(diccionario_dfs):
    """
    Aplica las reglas de limpieza y estandarización a cada dataset
    del diccionario de sedes (BCN, MAD, SEV, VAL).
    
    Retorna un nuevo diccionario con ls DataFrames totalmente limpios
    """
    dict_limpio = {}
    for sede, df in diccionario_dfs.items():
        print(f"\n==========================================")
        print(f"  Iniciando limpieza de la sede: {sede}")
        print(f"==========================================")

        df_clean = df.copy()

        # --------------------------------------------------
        # 1. Columna: producto_id (Garantizar identificador único)
        # --------------------------------------------------
        # Eliminar filas con ID nulo o vacío
        df_clean = df_clean.dropna(subset=['producto_id'])
        # Eliminar duplicados exactos basándonos en producto_id
        df_clean = df_clean.drop_duplicates(subset=['producto_id'], keep='first')
        
        # --------------------------------------------------
        # 2. Columna: nombre
        # --------------------------------------------------
        # Imputar nulos con valor por defecto
        df_clean['nombre'] = df_clean['nombre'].fillna('Producto no especificado')
        # Limpieza de espacios iniciales y finales
        df_clean['nombre'] = df_clean['nombre'].astype(str).str.strip()
        
        # --------------------------------------------------
        # 3. Columna: categoria
        # --------------------------------------------------
        df_clean['categoria'] = df_clean['categoria'].fillna('Sin categoría')
        df_clean['categoria'] = df_clean['categoria'].astype(str).str.strip().str.capitalize()
        
        # --------------------------------------------------
        # 4. Columna: proveedor
        # --------------------------------------------------
        df_clean['proveedor'] = df_clean['proveedor'].fillna('Proveedor no asignado')
        df_clean['proveedor'] = df_clean['proveedor'].astype(str).str.strip()
        
        # --------------------------------------------------
        # 5. Columna: precio_unitario
        # --------------------------------------------------
        # Convertir a numérico (fuerza NaN en espacios en blanco o textos corruptos como en SEV)
        df_clean['precio_unitario'] = pd.to_numeric(df_clean['precio_unitario'], errors='coerce')
        # Corregir valores negativos convirtiéndolos a su valor absoluto (módulo)
        df_clean['precio_unitario'] = df_clean['precio_unitario'].abs()
        # Imputar valores nulos con la mediana de la categoría correspondientes (o mediana global si es NaN)
        mediana_precio = df_clean['precio_unitario'].median()
        df_clean['precio_unitario'] = df_clean['precio_unitario'].fillna(mediana_precio)
        # Redondear a 2 decimales para formato moneda
        df_clean['precio_unitario'] = df_clean['precio_unitario'].round(2)
        
        # --------------------------------------------------
        # 6. Columna: stock_actual
        # --------------------------------------------------
        # Forzar tipo numérico y rellenar nulos con 0
        df_clean['stock_actual'] = pd.to_numeric(df_clean['stock_actual'], errors='coerce').fillna(0)
        # Convertir valores negativos a 0
        df_clean['stock_actual'] = df_clean['stock_actual'].apply(lambda x: max(0, int(x)))
        df_clean['stock_actual'] = df_clean['stock_actual'].astype(int)
        
        # --------------------------------------------------
        # 7. Columna: stock_minimo
        # --------------------------------------------------
        df_clean['stock_minimo'] = pd.to_numeric(df_clean['stock_minimo'], errors='coerce')
        mediana_stock_min = df_clean['stock_minimo'].median()
        df_clean['stock_minimo'] = df_clean['stock_minimo'].fillna(mediana_stock_min)
        df_clean['stock_minimo'] = df_clean['stock_minimo'].apply(lambda x: max(0, int(x))).astype(int)
        
        # --------------------------------------------------
        # 8. Columna: almacen
        # --------------------------------------------------
        # Homogeneizar el nombre de la sede
        df_clean['almacen'] = df_clean['almacen'].astype(str).str.strip().str.capitalize()
        
        # --------------------------------------------------
        # 9. Columna: fecha_ultima_entrada
        # --------------------------------------------------
        # Convertir a formato datetime de pandas
        df_clean['fecha_ultima_entrada'] = pd.to_datetime(df_clean['fecha_ultima_entrada'], errors='coerce')
        # Para nulos en fecha, rellenar con NaT o la fecha más reciente/por defecto si se prefiere
        
        # --------------------------------------------------
        # 10. Columna: estado
        # --------------------------------------------------
        # Limpieza inicial de texto
        df_clean['estado'] = df_clean['estado'].astype(str).str.strip().str.lower()
        # Mapeo de corrección para homologar categorías sucias ('X', 'retirado?', 'pendiente', etc.)
        mapa_estados = {
            'activo': 'activo',
            'descatalogado': 'descatalogado',
            'retirado?': 'descatalogado',
            'x': 'descatalogado',
            'pendiente': 'activo'
        }
        df_clean['estado'] = df_clean['estado'].map(lambda x: mapa_estados.get(x, 'activo'))
        
        # Guardar DataFrame limpio en el nuevo diccionario
        dict_limpio[sede] = df_clean
        
        print(f"✅ Sede {sede} limpiada correctamente.")
        print(f"   Filas iniciales: {len(df)} | Filas tras eliminar duplicados/nulos en ID: {len(df_clean)}")
        print(f"   Nulos restantes totales: {df_clean.isnull().sum().sum()}")
        
    return dict_limpio

dict_inventarios = {
    'BCN': diccionario_inventario["inventario_BCN.csv"], # o .csv según la clave exacta
    'MAD': diccionario_inventario["inventario_MAD.csv"],
    'SEV': diccionario_inventario["inventario_SEV.json"],
    'VAL': diccionario_inventario["inventario_VAL.json"]
}

# Ejecución de la función sobre el diccionario de sedes
# Nota: asegúrate de pasarle tu diccionario original (ej. dict_inventarios)
dict_inventarios_limpios = limpiar_inventario(dict_inventarios)



  Iniciando limpieza de la sede: BCN
✅ Sede BCN limpiada correctamente.
   Filas iniciales: 750 | Filas tras eliminar duplicados/nulos en ID: 750
   Nulos restantes totales: 0

  Iniciando limpieza de la sede: MAD
✅ Sede MAD limpiada correctamente.
   Filas iniciales: 750 | Filas tras eliminar duplicados/nulos en ID: 742
   Nulos restantes totales: 5

  Iniciando limpieza de la sede: SEV
✅ Sede SEV limpiada correctamente.
   Filas iniciales: 750 | Filas tras eliminar duplicados/nulos en ID: 743
   Nulos restantes totales: 5

  Iniciando limpieza de la sede: VAL
✅ Sede VAL limpiada correctamente.
   Filas iniciales: 750 | Filas tras eliminar duplicados/nulos en ID: 750
   Nulos restantes totales: 0


### 🔗 Paso 4: Unificación e Integración del Dataset Maestro

Una vez ejecutada la limpieza individual sobre los cuatro datasets del diccionario, procedemos a la **integración vertical** de los datos para generar una base de datos global y centralizada.

* **Trazabilidad:** Se mantiene la columna `almacen` para identificar el origen de cada registro (`Barcelona`, `Madrid`, `Sevilla`, `Valencia`).
* **Concatenación:** Hacemos uso de `pd.concat()` reindexando las filas con `ignore_index=True`.
* **Exportación:** Guardamos el resultado consolidado en el archivo `inventario_unificado_limpio.csv`.

In [109]:
# Unificar los 4 datasets limpios en un único DataFrame maestro
df_maestro_unificado = pd.concat(dict_inventarios_limpios.values(), ignore_index=True)

# Guardar el resultado consolidado
df_maestro_unificado.to_csv('inventario_unificado_limpio.csv', index=False)

print(f"¡Dataset unificado creado con éxito! Total de registros consolidados: {len(df_maestro_unificado)}")
display(df_maestro_unificado.head(10))

¡Dataset unificado creado con éxito! Total de registros consolidados: 2985


,producto_id,nombre,categoria,proveedor,precio_unitario,stock_actual,stock_minimo,almacen,fecha_ultima_entrada,estado
0,BCN-00001,Mando Max-395,Gaming,TechNova Iberia,767.14,83,38,Barcelona,2025-08-25,activo
1,BCN-00002,Ratón Pro-182,Informática,GigaSupply,1111.89,107,32,Barcelona,2026-09-22,activo
2,BCN-00003,Smartphone Pro-267,Telefonía,TechNova Iberia,750.46,58,16,Barcelona,2025-07-15,activo
3,BCN-00004,Adaptador Core-164,Accesorios,OmniTech,278.63,74,20,Barcelona,2025-10-25,activo
4,BCN-00005,Funda Go-139,Telefonía,IberDevices,767.67,118,34,Barcelona,2025-09-02,activo
5,BCN-00006,Webcam Neo-939,Informática,GigaSupply,1170.72,26,13,Barcelona,2026-06-13,activo
6,BCN-00007,Mini PC Lite-693,Informática,DigitalCore,62.60,17,21,Barcelona,2025-07-24,descatalogado
7,BCN-00008,Smartphone Neo-134,Telefonía,OmniTech,794.02,74,34,Barcelona,2025-12-02,activo
8,BCN-00009,Auricular TWS Pro-174,Telefonía,OmniTech,1359.28,91,31,Barcelona,2025-09-29,activo
9,BCN-00010,Cable USB-C Max-207,Telefonía,IberDevices,933.52,40,35,Barcelona,2026-01-29,activo
